## 1. Import Libraries

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set(style="whitegrid")


In [ ]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep="\t", header=None, names=["label", "message"])
df.head()


In [ ]:
print("Shape:", df.shape)
print(df["label"].value_counts())


## 3. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="label", palette="Set2")
plt.title("Spam vs Ham Message Count")
plt.show()


In [ ]:
df["message_length"] = df["message"].apply(len)

plt.figure(figsize=(8,5))
sns.histplot(data=df, x="message_length", hue="label", bins=50, kde=True)
plt.title("Message Length Distribution by Label")
plt.xlabel("Message Length (characters)")
plt.show()


## 4. Text Preprocessing

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_message"] = df["message"].apply(clean_text)
df[["message", "clean_message"]].head()


In [ ]:
df["label_num"] = df["label"].map({"ham": 0, "spam": 1})

X = df["clean_message"]
y = df["label_num"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


## 5. Feature Extraction with TF-IDF

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", max_features=3000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF matrix shape (train):", X_train_tfidf.shape)


## 6. Train Models

In [ ]:
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)


In [ ]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)


## 7. Evaluate Model Performance

In [ ]:
def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    print(f"===== {name} =====")
    print(f"Accuracy: {acc:.4f}\n")
    print(classification_report(y_true, y_pred, target_names=["ham", "spam"]))
    return acc

acc_nb = evaluate("Naive Bayes", y_test, nb_preds)
acc_lr = evaluate("Logistic Regression", y_test, lr_preds)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))

for ax, preds, name in zip(axes, [nb_preds, lr_preds], ["Naive Bayes", "Logistic Regression"]):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["ham","spam"], yticklabels=["ham","spam"], ax=ax)
    ax.set_title(f"Confusion Matrix - {name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


## 8. Test on Custom Messages

In [ ]:
def predict_message(msg, model=lr_model):
    cleaned = clean_text(msg)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    return "SPAM" if pred == 1 else "HAM"

test_messages = [
    "Congratulations! You've won a free iPhone. Click here to claim now!",
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT: Your account has been suspended. Verify immediately.",
    "Don't forget to bring the documents for the meeting."
]

for msg in test_messages:
    print(f"Message: {msg}\nPrediction: {predict_message(msg)}\n")
